# Yahoo Finance Historical Data Pipeline

This notebook is a **thin orchestrator** that delegates logic to modular packages:

| Module | Responsibility |
| --- | --- |
| `config/settings.py` | Table names, ticker lists, retry settings, logging |
| `utils/delta_helpers.py` | Delta table operations (check tickers, get latest dates, merge) |
| `transforms/yfinance_transforms.py` | Yahoo Finance download + transform (with retry) |

**Pipeline steps:**
1. Install dependencies
2. Import modules
3. Identify missing / stale tickers
4. Download & transform (incremental or full)
5. Merge (upsert) into Delta table

In [0]:
%pip install yfinance --quiet
dbutils.library.restartPython()

In [0]:
import sys
import os

# Ensure the src directory is on the Python path
src_dir = os.path.dirname(os.path.abspath(globals().get("__file__", ".")))
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Import pipeline modules
from config.settings import ALL_TICKERS, FULL_TABLE_NAME, get_logger
from utils.delta_helpers import get_missing_tickers, get_latest_dates, merge_to_delta
from transforms.yfinance_transforms import download_and_transform

import pandas as pd

logger = get_logger("yfinance_pipeline")
logger.info(f"Pipeline initialized | Table: {FULL_TABLE_NAME} | Tickers: {ALL_TICKERS}")

In [0]:
# Identify what needs to be downloaded
missing_tickers = get_missing_tickers(ALL_TICKERS, FULL_TABLE_NAME)
latest_dates = get_latest_dates(FULL_TABLE_NAME)
existing_tickers = [t for t in ALL_TICKERS if t in latest_dates]

logger.info(f"New tickers (full download): {missing_tickers}")
logger.info(f"Existing tickers (incremental): {existing_tickers}")

In [0]:
# Download and transform data
all_pandas_dfs: list[pd.DataFrame] = []

# 1. Full history for new tickers
for ticker in missing_tickers:
    pdf = download_and_transform(ticker, start_date=None)
    if not pdf.empty:
        all_pandas_dfs.append(pdf)

# 2. Incremental data for existing tickers
for ticker in existing_tickers:
    pdf = download_and_transform(ticker, start_date=latest_dates[ticker])
    if not pdf.empty:
        all_pandas_dfs.append(pdf)

logger.info(f"DataFrames collected: {len(all_pandas_dfs)}")

In [0]:
# Combine results into a Spark DataFrame
if all_pandas_dfs:
    combined_pandas = pd.concat(all_pandas_dfs, ignore_index=True)
    logger.info(f"Total new rows to merge: {len(combined_pandas)}")
    combined_df = spark.createDataFrame(combined_pandas)
    display(combined_df.limit(20))
else:
    logger.info("No new data to download. Everything is up to date.")
    combined_df = None

In [0]:
# Merge into Delta table
if combined_df is not None:
    merge_to_delta(combined_df, FULL_TABLE_NAME)
else:
    logger.info("Nothing to merge. Pipeline complete.")